주어진 파일 “gaia_sample.csv”에는 150pc 이내의 쌍성 후보들이 들어있다. 이 후보들 중 일부는 순수 쌍성이 아니고 얼마의 거리에 제3의 별이 있다. 또한, 일부는 두 별 사이의 거리가 너무 멀어 쌍성이 아닐 가능성이 크다.

주어진 파일 “GaiaSource150pc_coordinates.csv”은 150pc 이내에서 발견된 모든 source의 적경, 적위 좌표와 거리가 들어있다. 이 값들에 오차가 있으나 이 과제에서는 오차를 생략한다. 적경과 적위가 degree 단위 임에 유의하라.

(1) “gaia_sample.csv”에 있는 각 쌍성 후보의 첫번째 별로부터 3차원 공간에서 반경 5s이내에 있는 source의 (첫번째 별 포함) 수를 세는 코드를 짜라. 여기서 𝑠는 쌍성 후보 두 별사이의 하늘에서 거리이다. 단위는 kau 즉 1000au로 주어졌다. 편의상 각이 작을 때 호를 직선으로 가정하라.

In [3]:
import numpy as np

# 구면 좌표를 3차원 직교 좌표로 변환하는 함수
def spherical_to_cartesian(ra_deg, dec_deg, distance_pc):
    """
    구면 좌표(적경, 적위, 거리)를 3차원 직교 좌표로 변환
    ra_deg, dec_deg: 적경과 적위 (도 단위)
    distance_pc: 거리 (pc 단위)
    """
    ra_rad = np.radians(ra_deg)
    dec_rad = np.radians(dec_deg)

    # 3차원 직교 좌표 계산
    x = distance_pc * np.cos(dec_rad) * np.cos(ra_rad)
    y = distance_pc * np.cos(dec_rad) * np.sin(ra_rad)
    z = distance_pc * np.sin(dec_rad)

    return np.column_stack((x, y, z)) if isinstance(ra_deg, np.ndarray) else np.array([x, y, z])

if __name__ == "__main__":

    # binary_candidates 데이터 로드
    binary_candidates = np.genfromtxt("gaia_sample.csv", delimiter=',',skip_header=1,
    dtype=[('source_id1', 'i8'), ('source_id2', 'i8'),('s_kau', 'f8'), ('d1_pc', 'f8'), ('d2_pc', 'f8'),('ra1_deg', 'f8'), ('dec1_deg', 'f8'),('ra2_deg', 'f8'), ('dec2_deg', 'f8')])

    # all_sources 데이터 로드
    all_sources = np.genfromtxt("GaiaSource150pc_coordinates.csv", delimiter=',', skip_header=1,
    dtype=[('source_id', 'i8'), ('ra_deg', 'f8'), ('dec_deg', 'f8'), ('d_pc', 'f8')])

print(f"쌍성 후보의 수: {len(binary_candidates):,}")
print(f"전체 소스의 수: {len(all_sources):,}")


counts = np.zeros(len(binary_candidates), dtype=int)

    # 모든 소스의 3D 좌표 계산
all_source_coords = spherical_to_cartesian(all_sources['ra_deg'], all_sources['dec_deg'], all_sources['d_pc'])

     # 각 쌍성 후보에 대해 반복
for i, candidate in enumerate(binary_candidates):
        # 첫 번째 별의 3D 좌표 계산
    star1_coords = spherical_to_cartesian(candidate['ra1_deg'], candidate['dec1_deg'], candidate['d1_pc'])

    search_radius_pc = 5 * candidate['s_kau'] * 0.00485 ##pc변환

        # 모든 소스까지의 거리 계산 (3D 유클리드 거리 공식)
    distances = np.sqrt(np.sum((all_source_coords - star1_coords)**2, axis=1))

        # 반경 내에 있는 소스 수 계산
    counts[i] = np.sum(distances <= search_radius_pc)
    chance_alignment = np.sum(counts == 1)
    pure_binary = np.sum(counts == 2)
    multiple_system = np.sum(counts >= 3)


쌍성 후보의 수: 3,169
전체 소스의 수: 2,177,122


(2) 위 (1)의 코드를 사용하여 결과를 얻어서 숫자가 2이면 순수한 쌍성, 3 이상이면 다중성, 1이면 우연 겹침(chance alignment)로 해석하자. 이 세가지 경우의 수를 각각 구하라.

In [4]:
print("\n결과 분석:")
print(f"우연 겹침 (소스 수 = 1): {chance_alignment}")
print(f"순수한 쌍성 (소스 수 = 2): {pure_binary}")
print(f"다중성 (소스 수 >= 3): {multiple_system}")


결과 분석:
우연 겹침 (소스 수 = 1): 2597
순수한 쌍성 (소스 수 = 2): 564
다중성 (소스 수 >= 3): 8
